# Memory: remember, recall, forget

Goal: give an agent a durable memory with `MemoryExtension`, let a scripted
model store a fact with the `remember` tool, recall that fact into a later
run's context, and then forget it.

Trust: [T2](../../docs/site/security-trust-levels.md) callback. Network: none.

`MemoryExtension` is a factory for three native handles that plug into the
same factory parameters the trusted Python ports use:

| Handle | Parameter | Role |
| --- | --- | --- |
| `memory.toolset()` | `toolsets=[...]` | `remember`, `search_memory`, `inspect_memory`, `forget_memory`, `correct_memory` |
| `memory.context_provider()` | `context_providers=[...]` | recalls matching records into the model request |
| `memory.observer()` | `observers=[...]` | captures marked candidate memories from run events |

All three share one store and one scope, so one extension can back several
agents.

## Open a durable store

`MemoryExtension.sqlite(path=...)` creates the file and its schema on first
open; `MemoryExtension.in_process(...)` is the ephemeral equivalent. The
scope is fixed here: `tenant` sets the outer boundary, and `user` /
`agent` / `workspace` narrow it further.

The recall provider only contributes when the extension's `tenant` equals
the tenant scope of the run asking for it — that check is what keeps one
tenant's memories out of another's context. `tenant` therefore defaults to
`"python-local"`, the scope `Agent.run` and `Agent.start` bind runs to, so
the cell below passes no tenant at all. Pass one explicitly when the
recalling runs execute on a lane, using that session's tenant scope; a
mismatch fails recall with `context_contribution_invalid`.

In [ ]:
import tempfile
from pathlib import Path

import finstack_ai

workdir = Path(tempfile.mkdtemp(prefix="finstack-memory-"))
memory = finstack_ai.MemoryExtension.sqlite(
    path=str(workdir / "memory.db"),
    user="analyst-1",
    manage=True,
)
print(memory.tenant, memory.toolset().component, memory.toolset().tool_count)
assert memory.toolset().tool_count == 5

## Policy flags gate the tool surface

`read` covers `search_memory` and `inspect_memory`, `write` covers
`remember`, and `manage` covers `forget_memory` and `correct_memory`. The
defaults are `read=True, write=True, manage=False`; only the permitted
tools are ever offered to the model.

In [ ]:
read_only = finstack_ai.MemoryExtension.in_process(write=False)
default_policy = finstack_ai.MemoryExtension.in_process()
print(read_only.toolset().tool_count, default_policy.toolset().tool_count)
assert (read_only.toolset().tool_count, default_policy.toolset().tool_count) == (2, 3)

## A scripted model that remembers

The first model turn calls `remember` with keywords and a body; the second
turn reads the tool result and finishes. Scope is never a tool argument —
it is bound to the extension, so a model cannot write outside it.

In [ ]:
from typing import Any

BODY = "The quarterly close deadline is the fifth business day."
write_calls = 0


async def writing_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context, request
    global write_calls
    write_calls += 1
    if write_calls == 1:
        return {
            "text": "",
            "completion_id": "memory-1",
            "tool_calls": [
                {
                    "name": "remember",
                    "arguments": {
                        "id": "close-deadline",
                        "keywords": ["quarterly", "close", "deadline"],
                        "body": BODY,
                    },
                }
            ],
        }
    return {"text": "Stored.", "completion_id": "memory-2"}


writer = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        writing_model,
        component="notebook.model.memory-writer",
        provider="notebook",
        model="notebook-model",
        context_window_tokens=32_768,
    ),
    [memory.toolset()],
    "Remember durable facts the operator states.",
)
print((await writer.run("remember the close deadline")).text, write_calls)
assert write_calls == 2

## Recall in a later run

A second agent registers only the recall provider. Before the model is
called, the provider searches the store with the run's user input and
contributes matching records as bounded reference context items. The
callback below captures the request so we can see what the model saw.

In [ ]:
seen: list[dict[str, Any]] = []


async def reading_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context
    seen.append(request)
    return {"text": "Answered from memory.", "completion_id": "memory-3"}


def recalled_texts(request: dict[str, Any]) -> list[str]:
    texts: list[str] = []
    for message in request.get("messages", []):
        for block in message.get("content", []):
            text = block.get("text", "")
            if text.endswith("[memory]"):
                texts.append(text)
    return texts


reader = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        reading_model,
        component="notebook.model.memory-reader",
        provider="notebook",
        model="notebook-model",
        context_window_tokens=32_768,
    ),
    context_providers=[memory.context_provider(max_hits=4)],
)
print((await reader.run("when is the quarterly close deadline?")).text)
print(recalled_texts(seen[0]))
assert any(BODY in text for text in recalled_texts(seen[0]))

## Forget

`forget_memory` tombstones a record. The store then excludes it from every
search, so the next run recalls nothing. `correct_memory` is the other
`manage` tool: it supersedes a record with a corrected replacement instead
of dropping it.

In [ ]:
forget_calls = 0


async def forgetting_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context, request
    global forget_calls
    forget_calls += 1
    if forget_calls == 1:
        return {
            "text": "",
            "completion_id": "memory-4",
            "tool_calls": [
                {"name": "forget_memory", "arguments": {"id": "close-deadline"}}
            ],
        }
    return {"text": "Forgotten.", "completion_id": "memory-5"}


forgetter = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        forgetting_model,
        component="notebook.model.memory-forgetter",
        provider="notebook",
        model="notebook-model",
        context_window_tokens=32_768,
    ),
    [memory.toolset()],
    "Forget records the operator retracts.",
)
print((await forgetter.run("forget the close deadline")).text)

seen.clear()
print((await reader.run("when is the quarterly close deadline?")).text)
print(recalled_texts(seen[0]))
assert recalled_texts(seen[0]) == []

## What the observer adds

`memory.observer()` goes in `observers=[...]`. It watches run events and
writes candidate memories the rule-based extractor finds — text explicitly
marked for retention — into the same store, so capture does not depend on
the model calling `remember`. It is registered exactly like the other two
handles:

```python
agent = await finstack_ai.Agent.from_python(
    model,
    [memory.toolset()],
    context_providers=[memory.context_provider()],
    observers=[memory.observer()],
)
```

The same three parameters exist on every linked provider factory
(`Agent.openai`, `Agent.anthropic`, `Agent.ollama`, `Agent.openrouter`,
`Agent.gateway`, `Agent.e2b_sandbox`), so a memory-backed agent is one
argument list away from a real provider.

In [ ]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("cleaned", workdir)